# BTC Perpetual Futures: Wavelet Jump Classification & Probabilistic Price Forecasting

A comprehensive research pipeline for detecting price jumps, classifying them using wavelet scattering, and predicting outcomes with uncertainty calibration.

## 🚀 System Overview
1. **Jump Detection**: Statistical Gumbel-threshold method with intraday seasonality.
2. **Feature Engineering**: 63-feature matrix (29 Wavelet, CVD, OI, Volume Profile).
3. **Regimes**: Macro HMM (4-states) and Micro Efficiency regimes.
4. **ML Model Stack**: Dual LightGBM stack for Jump Type -> Outcome.
5. **Uncertainty**: GP-based gating for signal filtering.

## 1. Setup & Imports

In [1]:
!pip install hmmlearn lightgbm scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 5.2 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pywt
import lightgbm as lgb
import scipy.stats as stats
from typing import List, Dict, Tuple
from tqdm.notebook import tqdm
from hmmlearn.hmm import GaussianHMM
from scipy.stats import linregress
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.ensemble import IsolationForest
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# Settings
class CONFIG:
    SYMBOL = "BTCUSDT"
    # NEW — replace with these 4 lines:
    TRAIN_START = "2021-01-01"
    TRAIN_END   = "2022-12-31"
    VAL_START   = "2023-01-01"
    VAL_END     = "2023-06-30"

    JUMP_SIGMA_WINDOW = 120
    JUMP_THRESHOLD = 4.0
    JUMP_CLUSTER_GAP = 10

    WAVELET_J = 6
    WAVELET_NAME = 'cmor1.5-1.0'

    MACRO_HMM_STATES = 2

    LABEL_ENDO_VOL_BUILDUP = 1.5
    LABEL_ENDO_SCAT_ASYM   = 0.0
    LABEL_ANTI_VOL_BUILDUP = 0.8
    LABEL_ANTI_TREND_ALIGN = 0.55

    BUCKET_STRONG_UP   =  0.493
    BUCKET_MILD_UP     =  0.112
    BUCKET_FLAT_LO     = -0.131
    BUCKET_MILD_DOWN   = -0.512

np.random.seed(42)
print("Environment initialized.")

Environment initialized.


## 2. Data loading
We load klines from `dataset/Market Data` and funding from `dataset/BTCUSDT-fundingRate-2026-03`.

In [4]:
def load_data():
    klines_path = "/content/Market Data"
    funding_path = "/content/Funding Rate"

    k_files = sorted([f for f in os.listdir(klines_path) if f.endswith(".csv")])
    k_list = []
    for f in tqdm(k_files, desc="Klines"):
        tmp = pd.read_csv(os.path.join(klines_path, f), header=0, names=[
            "open_time", "open", "high", "low", "close", "volume",
            "close_time", "quote_vol", "n_trades", "taker_buy_base", "taker_buy_quote", "ignore"
        ])
        tmp["open_time"] = pd.to_datetime(tmp["open_time"], unit="ms", utc=True)
        k_list.append(tmp.set_index("open_time"))

    df = pd.concat(k_list).sort_index()
    df["delta"] = df["taker_buy_base"] - (df["volume"] - df["taker_buy_base"])
    df["cvd"] = df["delta"].cumsum()

    f_list = []
    for root, _, files in os.walk(funding_path):
        for f in files:
            if f.endswith(".csv"):
                tmp = pd.read_csv(os.path.join(root, f))
                tmp["calc_time"] = pd.to_datetime(tmp["calc_time"], unit="ms", utc=True)
                f_list.append(tmp.rename(columns={"last_funding_rate": "funding_rate"}).set_index("calc_time")[["funding_rate"]])

    df_f = pd.concat(f_list).sort_index() if f_list else pd.DataFrame(columns=["funding_rate"])
    return df.astype(float), df_f

df_klines, df_funding = load_data()
print(f"Loaded {len(df_klines)} bars and {len(df_funding)} funding entries.")

Klines:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded 2103840 bars and 4383 funding entries.


## 3. Jump Detection Model

In [5]:
def detect_jumps(df):
    rets = np.log(df["close"] / df["close"].shift(1))
    sigma = rets.rolling(CONFIG.JUMP_SIGMA_WINDOW).std()
    f = rets.abs().groupby(rets.index.hour).transform('mean') / (rets.abs().mean() + 1e-12)

    score = rets / (f * sigma + 1e-12)
    is_jump = score.abs() > CONFIG.JUMP_THRESHOLD

    jumps = df[is_jump].copy()
    jumps["direction"] = np.sign(score[is_jump])

    # Cluster filter
    filtered = []
    last = None
    for ts, row in jumps.iterrows():
        if last is None or (ts - last).total_seconds() / 60 > CONFIG.JUMP_CLUSTER_GAP:
            filtered.append(row)
            last = ts
    return pd.DataFrame(filtered)

df_jumps = detect_jumps(df_klines)
print(f"Detected {len(df_jumps)} price jumps.")

Detected 8989 price jumps.


## 4. Market Regime Detection (Macro & Micro)

In [6]:
class RegimeDetector:
    def __init__(self):
        self.hmm = GaussianHMM(n_components=CONFIG.MACRO_HMM_STATES, covariance_type="diag", n_iter=500, tol=1e-4, random_state=42)

    def fit_macro(self, df, funding_df):
      daily = df.resample('1D').agg({'close': 'last', 'volume': 'sum'})
      daily['log_ret']    = np.log(daily['close'] / daily['close'].shift(1))
      daily['realized_vol'] = daily['log_ret'].rolling(20).std()
      daily['ret_5d']     = daily['log_ret'].rolling(5).sum()  # 5-day momentum

      f_avg = funding_df.resample('1D').mean()
      daily['funding']    = f_avg['funding_rate'].reindex(
                              daily.index, method='ffill').fillna(0)

    # Use 4 features instead of 2 — gives HMM more to separate states on
      data = daily[['realized_vol', 'funding', 'ret_5d', 'log_ret']].dropna()
      self.hmm.fit(data.values)

      states = pd.Series(self.hmm.predict(data.values), index=data.index)

    # Verify states are not degenerate
      state_counts = states.value_counts()
      if len(state_counts) < 2:
        print("WARNING: HMM collapsed to 1 state. Reducing n_components to 2.")
        self.hmm = GaussianHMM(n_components=2, covariance_type='diag',
                               n_iter=500, tol=1e-4, random_state=42)
        self.hmm.fit(data.values)
        states = pd.Series(self.hmm.predict(data.values), index=data.index)

      return states

    def detect_micro(self, df_slice, cvd_slope, cvd_div):
        # Efficiency Ratio
        net_move = abs(df_slice["close"].iloc[-1] - df_slice["close"].iloc[-15])
        total_path = df_slice["close"].diff().abs().iloc[-14:].sum()
        er = net_move / (total_path + 1e-12)

        if er > 0.35 and abs(cvd_slope) > 0.01: return 1 # Trending
        if er > 0.35 and cvd_div < 0.3: return 2 # Exhaustion
        return 0 # Balanced

regime_detector = RegimeDetector()
daily_regimes = regime_detector.fit_macro(df_klines, df_funding)
# Check convergence
monitor = regime_detector.hmm.monitor_
print(f"HMM converged: {monitor.converged}")
print(f"HMM iterations: {len(monitor.history)}")
print(f"Macro regime value counts:\n{daily_regimes.value_counts().sort_index()}")

HMM converged: True
HMM iterations: 46
Macro regime value counts:
0    1097
1     344
Name: count, dtype: int64


## 5. Feature Extraction & Matrix Assembly

In [7]:
class FeatureExtractor:
    # NEW — replace the entire extract_wavelet method body with this
    def extract_wavelet(self,window_rets):
      PRE_BARS = 30   # number of pre-jump bars
      JUMP_IDX = PRE_BARS  # index of the jump bar within the window

      local_std = np.std(window_rets) + 1e-12
      x = window_rets / local_std

      feats = {}

      # First-order CWT coefficients at the jump bar (index 30)
      for j in range(1, CONFIG.WAVELET_J + 1):
          scale = 2 ** j
          coeffs, _ = pywt.cwt(x, [scale], CONFIG.WAVELET_NAME)
          feats[f'wavelet_real_j{j}'] = float(np.real(coeffs[0, JUMP_IDX]))
          feats[f'wavelet_imag_j{j}'] = float(np.imag(coeffs[0, JUMP_IDX]))

    # Second-order scattering at the jump bar
    # Now that post-jump bars exist, the scattering coefficient at JUMP_IDX
    # has both pre and post context — imaginary part will be negative when
    # pre-jump volatility dominates (endogenous) and positive when
    # post-jump volatility dominates (exogenous)
      # REPLACE the vol_profile + scattering loop with these direct asymmetry measures:

# Quarter-by-quarter volatility profile (captures acceleration shape)
      q1 = np.std(x[0:8])   + 1e-12
      q2 = np.std(x[8:16])  + 1e-12
      q3 = np.std(x[16:24]) + 1e-12
      q4 = np.std(x[24:30]) + 1e-12

      feats['vol_q1_q4_ratio']    = float(q4 / q1)   # late vs early buildup
      feats['vol_accel_midpoint']  = float(q3 / q2)   # acceleration in middle section
      feats['vol_convexity']       = float((q1 + q4) / (q2 + q3 + 1e-12))
      # vol_convexity > 1 = U-shape (volatile early + late, quiet middle) = anticipatory
      # vol_convexity < 1 = ramp shape (builds monotonically) = endogenous

      # Wavelet imaginary at multiple time points (not just JUMP_IDX)
      # Spread of imaginary parts shows asymmetry across the window
      imag_values = []
      for j in range(1, CONFIG.WAVELET_J + 1):
          scale  = 2 ** j
          coeffs, _ = pywt.cwt(x, [scale], CONFIG.WAVELET_NAME)
          # Sample at 3 points: early, mid, jump
          early_imag = float(np.imag(coeffs[0, 10]))
          mid_imag   = float(np.imag(coeffs[0, 20]))
          jump_imag  = float(np.imag(coeffs[0, JUMP_IDX]))
          feats[f'wavelet_real_j{j}'] = float(np.real(coeffs[0, JUMP_IDX]))
          feats[f'wavelet_imag_j{j}'] = jump_imag
          feats[f'wavelet_imag_trend_j{j}'] = float(jump_imag - early_imag)
          # Positive trend = imaginary growing toward jump = exogenous building
          # Negative trend = imaginary shrinking toward jump = endogenous releasing
          imag_values.append(jump_imag)

# Replace scat_imag_j{j1}_{j2} columns with simpler cross-scale correlations
      for j1 in range(1, CONFIG.WAVELET_J + 1):
          for j2 in range(j1 + 1, CONFIG.WAVELET_J + 1):
              c1, _ = pywt.cwt(x, [2**j1], CONFIG.WAVELET_NAME)
              c2, _ = pywt.cwt(x, [2**j2], CONFIG.WAVELET_NAME)
              # Cross-scale imaginary correlation — genuinely can be negative
              cross = float(np.imag(c1[0, JUMP_IDX]) * np.real(c2[0, JUMP_IDX])
                          - np.real(c1[0, JUMP_IDX]) * np.imag(c2[0, JUMP_IDX]))
              feats[f'scat_imag_j{j1}_{j2}'] = cross
    # vol_buildup_ratio: pre-jump vol vs post-jump vol
    # HIGH = post more volatile = exogenous (sudden shock, activity after)
    # LOW  = pre and post similar = endogenous (symmetric, self-generated)



      # CORRECTED — three features, clearly separated:

# 1. Internal pre-jump buildup (early half vs late half of pre-window)
#    Measures whether volatility was accelerating INTO the jump
      feats['vol_buildup_ratio'] = float(np.std(x[15:PRE_BARS]) / (np.std(x[:15]) + 1e-12))

# 2. Pre vs post contrast (the asymmetry signal)
#    LOW  = post not dramatically more volatile = endogenous (symmetric)
#    HIGH = post dominates = exogenous (sudden shock, activity after)
      feats['pre_post_vol_ratio'] = float(np.std(x[JUMP_IDX:]) / (np.std(x[:JUMP_IDX]) + 1e-12))

# 3. Trend alignment — pre-jump bars only
      feats['trend_alignment'] = float(np.mean(x[:PRE_BARS] > 0))

      return feats
    def extract_micro(self, df_slice, funding, funding_8h):
        cvd = df_slice["cvd"].iloc[-15:]
        price = df_slice["close"].iloc[-15:]
        s = linregress(range(len(cvd)), cvd.values).slope if len(cvd)>1 else 0
        d = float(np.corrcoef(price.values, cvd.values)[0, 1]) if len(cvd)>1 else 0

        feats = {
            "cvd_slope": s,
            "cvd_price_div": d,
            "delta_ratio": float(df_slice["delta"].iloc[-1] / (df_slice["volume"].iloc[-1] + 1e-12)),
            "funding": float(funding),
            "funding_change": float(funding - funding_8h),
            "micro_regime": float(regime_detector.detect_micro(df_slice, s, d))
        }
        return feats

extractor = FeatureExtractor()
feature_rows = []
for ts, row in tqdm(df_jumps.iterrows(), total=len(df_jumps)):
    idx = df_klines.index.get_loc(ts)
    if idx < 30: continue

    # Option B: extend window to include 3 post-jump bars for asymmetry detection
    # This adds a 3-minute wait before classification but gives the wavelet
    # the contrast it needs to distinguish pre vs post volatility
    POST_BARS = 3
    if idx + POST_BARS >= len(df_klines): continue

    pre_window = df_klines.iloc[idx-30 : idx]
    jump_slice = df_klines.iloc[idx-30 : idx+1]

    # Build extended window: 30 pre-jump bars + jump bar + 3 post-jump bars = 34 bars
    extended_window = df_klines.iloc[idx-30 : idx+1+POST_BARS]
    ext_rets = np.log(
        extended_window["close"] / extended_window["close"].shift(1)
    ).fillna(0).values * row["direction"]

    w_f = extractor.extract_wavelet(ext_rets)   # pass 34-bar window instead of 30

    f_val = df_funding.asof(ts)["funding_rate"] if not df_funding.empty else 0.0
    f_8h = df_funding.asof(ts - pd.Timedelta(hours=8))["funding_rate"] if not df_funding.empty else 0.0
    m_f = extractor.extract_micro(jump_slice, f_val, f_8h)

    # Macro Regime
    m_r = daily_regimes.asof(ts)
    m_f["macro_regime"] = float(m_r) if pd.notna(m_r) else 0.0

    ret_1h = np.nan
    if idx + 60 < len(df_klines):
        ret_1h = np.log(df_klines["close"].iloc[idx+60] / df_klines["close"].iloc[idx]) * 100

    feature_rows.append({**w_f, **m_f, "target_ret": ret_1h, "timestamp": ts})

fm = pd.DataFrame(feature_rows).set_index("timestamp").dropna()
print(f"Feature matrix build: {fm.shape}")

  0%|          | 0/8989 [00:00<?, ?it/s]

Feature matrix build: (8987, 47)


In [8]:
# Diagnose actual scat_imag distribution
scat_cols = [c for c in fm.columns if 'scat_imag' in c]
scat_means = fm[scat_cols].mean(axis=1)
print(f"scat_asym distribution:")
print(f"  min:    {scat_means.min():.6f}")
print(f"  max:    {scat_means.max():.6f}")
print(f"  mean:   {scat_means.mean():.6f}")
print(f"  median: {scat_means.median():.6f}")
print(f"  pct below 0.0: {(scat_means < 0.0).mean():.1%}")
print(f"  pct below -0.001: {(scat_means < -0.001).mean():.1%}")
print(f"vol_buildup_ratio > 1.5: {(fm['vol_buildup_ratio'] > 1.5).mean():.1%}")
print(f"Both conditions met: {((fm['vol_buildup_ratio'] > 1.5) & (scat_means < 0.0)).mean():.1%}")

scat_asym distribution:
  min:    -0.491277
  max:    0.209137
  mean:   -0.154562
  median: -0.150572
  pct below 0.0: 95.8%
  pct below -0.001: 95.6%
vol_buildup_ratio > 1.5: 25.2%
Both conditions met: 23.3%


## 6. Labeling & Modeling

In [9]:
# ── Step 1: Assign wavelet labels ─────────────────────────────────────────────
def assign_wavelet_label(row):
    # Guard — catches missing columns immediately with a clear message
    required = ['pre_post_vol_ratio', 'vol_buildup_ratio', 'trend_alignment']
    for col in required:
        if col not in row.index:
            raise KeyError(
                f"Column '{col}' missing from feature matrix. "
                f"Rerun cell 11 to rebuild fm before running this cell."
            )

    scat_cols      = [c for c in row.index if 'scat_imag' in c]
    scat_asym      = np.mean([row[c] for c in scat_cols])
    pre_post_ratio = row['pre_post_vol_ratio']

    if pre_post_ratio < 1.5 and scat_asym < 0.15:
        return 'endogenous'

    if row['vol_buildup_ratio'] < 0.8 and row['trend_alignment'] > 0.55:
        return 'anticipatory'

    return 'exogenous'

fm["jump_type"] = fm.apply(assign_wavelet_label, axis=1)

# ── Step 2: Verify label distribution ────────────────────────────────────────
label_dist   = fm["jump_type"].value_counts()
dominant_pct = label_dist.max() / label_dist.sum()
print(f"Jump type distribution:\n{label_dist}\n")
if len(label_dist) < 2 or dominant_pct > 0.95:
    raise ValueError(
        f"DEGENERATE LABELS: {label_dist.idxmax()} = {dominant_pct:.1%}\n"
        "Fix wavelet implementation or labeller thresholds before proceeding."
    )
print("Label distribution OK.\n")

# ── Step 3: Encode jump_type and add outcome BEFORE splitting ─────────────────
# jump_type_enc must be added to fm BEFORE train_df/val_df are created
# so both slices automatically inherit the column
le_jt = LabelEncoder()
fm['jump_type_enc'] = le_jt.fit_transform(fm['jump_type'])

fm["outcome"] = fm["target_ret"].apply(
    lambda r: "strong_up"   if r >= CONFIG.BUCKET_STRONG_UP  else
             ("mild_up"     if r >= CONFIG.BUCKET_MILD_UP    else
             ("flat"        if r >= CONFIG.BUCKET_FLAT_LO    else
             ("mild_down"   if r >= CONFIG.BUCKET_MILD_DOWN  else
              "strong_down"))))

# ── Step 4: Split AFTER all columns are added ─────────────────────────────────
train_df = fm[fm.index <= CONFIG.TRAIN_END]
val_df   = fm[(fm.index > CONFIG.TRAIN_END) & (fm.index <= CONFIG.VAL_END)]

print(f"Train: {len(train_df)} jumps | Val: {len(val_df)} jumps\n")

# ── Step 5: Define feature columns ───────────────────────────────────────────
# jump_type_enc is now included automatically since it's in fm
X_cols = [c for c in fm.columns if c not in
          ["target_ret", "jump_type", "outcome"]]

le_o = LabelEncoder()
y_tr = le_o.fit_transform(train_df["outcome"])
y_vl = le_o.transform(val_df["outcome"])

# ── Step 6: Train LightGBM with calibration ───────────────────────────────────
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

X_tr_raw  = train_df[X_cols].values
split_idx = int(len(X_tr_raw) * 0.8)

X_lgbm, X_calib = X_tr_raw[:split_idx],  X_tr_raw[split_idx:]
y_lgbm, y_calib = y_tr[:split_idx],      y_tr[split_idx:]

clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.03,
                          verbose=-1, class_weight='balanced')
clf.fit(X_lgbm, y_lgbm)

calibrated_clf = CalibratedClassifierCV(FrozenEstimator(clf), method='isotonic')
calibrated_clf.fit(X_calib, y_calib)

# ── Step 7: Get probabilities THEN apply flat filter ─────────────────────────
# probs must exist before the flat filter can use it
probs = calibrated_clf.predict_proba(val_df[X_cols].values)

mean_brier = np.mean([brier_score_loss((y_vl == i).astype(int), probs[:, i])
                      for i in range(5)])
print(f"Validation Mean Brier Score: {mean_brier:.4f}")

# ── Step 8: Flat class filter ─────────────────────────────────────────────────
flat_idx      = list(le_o.classes_).index('flat')
flat_prob     = probs[:, flat_idx]
ambiguous_mask = flat_prob > 0.35
clear_mask    = ~ambiguous_mask

print(f"\nAmbiguous (high flat prob): {ambiguous_mask.mean():.1%} of signals skipped")
print(f"Clear signals remaining:    {clear_mask.sum()} of {len(val_df)}")

if clear_mask.sum() > 20:
    clear_brier = np.mean([brier_score_loss(
        (y_vl[clear_mask] == i).astype(int), probs[clear_mask, i]
    ) for i in range(5)])
    print(f"Mean Brier on clear signals: {clear_brier:.4f}")
    print(f"Mean Brier on all signals:   {mean_brier:.4f}")

# ── Step 9: Per-class calibration ─────────────────────────────────────────────
print("\n── PER-CLASS CALIBRATION ──")
for i, name in enumerate(le_o.classes_):
    y_bin = (y_vl == i).astype(int)
    b     = brier_score_loss(y_bin, probs[:, i])
    try:
        frac_pos, mean_pred = calibration_curve(y_bin, probs[:, i], n_bins=8)
        ece = float(np.mean(np.abs(frac_pos - mean_pred)))
    except ValueError:
        ece = float("nan")
    print(f"  {name:<15}: Brier={b:.4f}  ECE={ece:.4f}  {'PASS' if b < 0.20 else 'FAIL'}")

# ── Step 10: Confidence deviation check ───────────────────────────────────────
print("\n── CONFIDENCE DEVIATION CHECK ──")
confidence = probs.max(axis=1)
predicted  = probs.argmax(axis=1)
correct    = (predicted == y_vl)
for lo, hi in [(0.4, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 1.0)]:
    mask = (confidence >= lo) & (confidence < hi)
    if mask.sum() < 20:
        continue
    actual_acc = correct[mask].mean()
    pred_conf  = confidence[mask].mean()
    deviation  = abs(actual_acc - pred_conf)
    edge = "EDGE" if deviation < 0.05 else "NO EDGE"
    print(f"  Conf {lo:.0%}-{hi:.0%}: n={mask.sum():4d}  "
          f"predicted={pred_conf:.1%}  actual={actual_acc:.1%}  "
          f"dev={deviation:.1%}  {edge}")

# ── Step 11: Jump type summary ────────────────────────────────────────────────
print("\n── JUMP TYPE DISTRIBUTION ──")
print(fm["jump_type"].value_counts().to_string())

Jump type distribution:
jump_type
exogenous       7737
endogenous       751
anticipatory     499
Name: count, dtype: int64

Label distribution OK.

Train: 2547 jumps | Val: 1266 jumps



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

Validation Mean Brier Score: 0.1512

Ambiguous (high flat prob): 58.1% of signals skipped
Clear signals remaining:    530 of 1266
Mean Brier on clear signals: 0.1561
Mean Brier on all signals:   0.1512

── PER-CLASS CALIBRATION ──
  flat           : Brier=0.2202  ECE=0.1304  FAIL
  mild_down      : Brier=0.1676  ECE=0.2383  PASS
  mild_up        : Brier=0.1940  ECE=0.1166  PASS
  strong_down    : Brier=0.0745  ECE=0.0271  PASS
  strong_up      : Brier=0.0996  ECE=0.1614  PASS

── CONFIDENCE DEVIATION CHECK ──
  Conf 40%-50%: n= 366  predicted=45.1%  actual=36.6%  dev=8.5%  NO EDGE
  Conf 50%-60%: n= 107  predicted=53.6%  actual=47.7%  dev=6.0%  NO EDGE

── JUMP TYPE DISTRIBUTION ──
jump_type
exogenous       7737
endogenous       751
anticipatory     499


## 7. Uncertainty Gating & Visuals

In [10]:
# Replace the entire cell 15 with this:
from sklearn.ensemble import RandomForestClassifier

# Change 1: Use ALL training data, not 800 subsampled
# Change 2: Train RF on jump_type labels (3-class), not outcome labels (5-class)
# 3-class is much easier — RF can find meaningful separation

rf_uncertainty = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

# Train on jump_type (3-class endogenous/exogenous/anticipatory)
# NOT on outcome (5-class) — jump type is more learnable
le_jt = LabelEncoder()
y_tr_jt = le_jt.fit_transform(train_df['jump_type'])
y_vl_jt = le_jt.transform(val_df['jump_type'])

# Use ALL training data — no subsampling
rf_uncertainty.fit(train_df[X_cols].values, y_tr_jt)
# In cell 16 — change the gating logic:
rf_probs = rf_uncertainty.predict_proba(val_df[X_cols].values)
rf_predicted_type = le_jt.inverse_transform(rf_probs.argmax(axis=1))

# LightGBM's highest-confidence outcome class
lgbm_top_class = le_o.classes_[probs.argmax(axis=1)]

# Agreement signal:
# endogenous RF type + flat/mild LightGBM outcome = consistent (mean reversion)
# exogenous RF type + strong directional LightGBM outcome = consistent (continuation)
# Disagreement = ambiguous = lower quality signal

def signals_agree(rf_type, lgbm_class):
    if rf_type == 'endogenous' and lgbm_class in ['mild_up', 'mild_down', 'flat']:
        return True   # endogenous = mean reverting = mild outcome consistent
    if rf_type == 'exogenous' and lgbm_class in ['strong_up', 'strong_down']:
        return True   # exogenous = continuation = strong outcome consistent
    if rf_type == 'anticipatory':
        return True   # anticipatory = always pass through
    return False      # disagreement = skip

agreement_mask = np.array([
    signals_agree(rf_type, lgbm_class)
    for rf_type, lgbm_class in zip(rf_predicted_type, lgbm_top_class)
])

print(f"Agreement filter: {agreement_mask.mean():.1%} of signals pass")
kept_probs  = probs[agreement_mask]
kept_actual = y_vl[agreement_mask]
if len(kept_actual) > 20:
    kept_brier = np.mean([brier_score_loss(
        (kept_actual==i).astype(int), kept_probs[:,i]
    ) for i in range(5)])
    print(f"Mean Brier on agreement signals: {kept_brier:.4f}")
    print(f"Mean Brier on all signals:       {mean_brier:.4f}")
    print(f"Improvement from agreement filter: {(mean_brier - kept_brier):.4f}")

Agreement filter: 20.3% of signals pass
Mean Brier on agreement signals: 0.1534
Mean Brier on all signals:       0.1512
Improvement from agreement filter: -0.0022
